# Fly-robot: a robot arm run by a piece of the real fly brain

The robot's camera is wired into the fly's **eye** (photoreceptors), its joints into the fly's
**body-sense** neurons (ascending neurons), and its motors are driven by the fly's **command**
neurons (descending neurons). In between: ~2,500 neurons on the real pathways through the
optic lobe and central brain (FlyWire connectome). Only the fly brain is trained (PPO).

The brain sees the tray through a 16x16 wrist camera (this frame + the previous one). It is
**not** told where the ball is, where the walls are or where the goal is.

**Tasks (`ENV`):**
- `tray_vision`: keep the ball on the tray (solved: ~250k steps on Kaggle)
- `maze_fixed3_vision`: one fixed 3x3 tilt maze, seen only through the camera (solved: 100 % from 1.1M steps
  with slow motors; 0 % on unseen mazes)
- `maze3_vision`: a new random maze every episode  <- current
- `maze3h_vision`: random mazes with holes

**Current setup (fresh run):** slow, smooth motors (`ACTION_RATE = 0.1`) + start-position
curriculum (`RANDOM_START = 0.5`). The first camera-maze run, with fast motors, rolled the ball to
the goal but never stopped it in the centre.

**Kaggle settings:** Internet ON. Accelerator: **None (CPU)**. About 2.7 h per 1M steps.
Use **Save & Run All (Commit)** for long runs. Checkpoints are saved every 50k steps.

In [ ]:
# ===== Settings =====
REPO_URL = "https://github.com/nihal4/connectome-tray-balance.git"
BRANCH = "main"

ENV = "maze3_vision"         # tray_vision -> maze_fixed3_vision -> maze3_vision -> maze3h_vision
TOTAL_STEPS = 3_000_000      # random mazes need more than one fixed maze (~8 h; resume after a timeout)
N_NEURONS = 2500             # brain size (eye 512 + body 64 + commands 64 + pathway neurons)
ACTION_RATE = 0.1            # slow, smooth motors: max joint change per step (0 = fast motors of the old runs).
                             # Changes the robot, so a run must keep the same value (do not change it on --resume)
RANDOM_START = 0.5           # maze curriculum: share of training episodes that start the ball at a random
                             # point on the route (evaluation always uses the real start). 0 = off
SEED = 0
DEVICE = "cpu"
RUN_NAME = f"{ENV}-connectome-real-s{SEED}" + ("-slow" if ACTION_RATE > 0 else "")

In [ ]:
import os, subprocess, sys, glob, shutil
WORK = "/kaggle/working"
REPO = f"{WORK}/connectome-tray-balance"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, REPO)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q mujoco gymnasium stable-baselines3 psutil imageio imageio-ffmpeg

## 1. Download the connectome + cell types, build the fly-robot brain

In [ ]:
!python scripts/download_connectome.py --out data/raw
!python scripts/build_flybrain.py --n {N_NEURONS}
BRAIN = sorted(glob.glob(f"data/flybrain_cam16_N*.npz"))[-1]
print("brain file:", BRAIN)

`eye_to_command_hops` shows how many synapses a signal crosses from the eye to the command
neurons inside this brain (3-5 for all of them).

## 2. Check the task (a hand-written controller can solve it, doing nothing cannot)

In [ ]:
!python scripts/check_mac.py --env {ENV} --subset {BRAIN}

## 3. Train the fly brain

In [ ]:
import re
KEEP = re.compile(r"^\[|Eval num|Episode length|Success rate|New best")
def train(env=ENV, extra=""):
    cmd = (f"{sys.executable} -u scripts/train.py --env {env} --policy connectome --subset {BRAIN} "
           f"--profile kaggle --total_steps {TOTAL_STEPS} --seed {SEED} --device {DEVICE} "
           f"--random_start {RANDOM_START if ENV.startswith('maze') else 0} --action_rate {ACTION_RATE} "
           f"--runs_dir {WORK}/runs --name {RUN_NAME} {extra}")
    p = subprocess.Popen(cmd.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if KEEP.search(line):
            print(line, end="", flush=True)
    if p.wait() != 0:
        print(f"!! exited with code {p.returncode}")

train()

Training prints an `Eval` block every 50k steps (deterministic run from the real start) and a
`[progress]` line (training episodes, including the easier curriculum starts). Return ~6 means
the ball reaches the goal but is not stopped in the centre; a solved episode scores ~+35-40.

To continue a run (after a timeout, or for more steps): raise `TOTAL_STEPS` and run
`train(extra="--resume")`. It continues from the last checkpoint in `runs/{RUN_NAME}`.

## 4. Results

In [ ]:
run = f"{WORK}/runs/{RUN_NAME}"
!python scripts/plot_runs.py {run} --out {WORK}/curves_{ENV}.png
from IPython.display import Image, Video, display
display(Image(f"{WORK}/curves_{ENV}.png"))

In [ ]:
# Tests it never saw in training: camera noise, blind (ball invisible), heavy ball, pushes
# (+ unseen mazes for maze_fixed3_vision)
!python scripts/evaluate.py --run {run} --robustness --episodes 10

## 5. Videos

One video per test. The camera image the brain receives is in the top-left corner.
Heavy ball: drawn dark grey. Pushes: the frame turns red on each shove.

In [ ]:
import numpy as np, imageio.v2 as imageio
os.environ["MUJOCO_GL"] = "egl"
from scripts.evaluate import load, make_norm, run_config, add_eye_view, run_env_kw
from ctb.envs import build_env

env_name = run_config(run)["env"]
TESTS = {"clean": {}, "heavy_ball": {"ball_mass_scale": 3.0}, "push": {"push_prob": 0.05}}
if env_name.startswith("maze_fixed3"):
    TESTS["unseen_maze"] = {"fixed_maze": False}

model = load(run)
vn = make_norm(run, {})          # same input normalisation as in training

def record_test(name, kw, seed=7):
    env = build_env(env_name, seed, render_mode="rgb_array", **run_env_kw(run), **kw)
    if name == "heavy_ball":
        env.model.geom_rgba[env.model.geom("ball").id] = (0.25, 0.25, 0.25, 1)
    obs, _ = env.reset(seed=seed)
    frames, pushes, info = [], 0, {}
    for t in range(env.max_steps):
        a, _ = model.predict(vn.normalize_obs(obs[None]), deterministic=True)
        obs, _, term, trunc, info = env.step(a[0])
        frame = env.render()
        if getattr(env, "vision", False):
            frame = add_eye_view(frame, env.camera_frame())
        if np.any(env.data.xfrc_applied[env.ball_body] != 0):
            pushes += 1
            frame = frame.copy(); frame[:8] = frame[-8:] = frame[:, :8] = frame[:, -8:] = (220, 40, 40)
        frames.append(frame)
        if term or trunc:
            break
    frames += [frames[-1]] * 25
    path = f"{WORK}/{env_name}_{name}.mp4"
    imageio.mimsave(path, frames, fps=50)
    env.close()
    solved = info.get("is_success", (not info.get("fell", False)) and t + 1 >= env.max_steps)
    print(f"{name:12s} {t + 1}/{env.max_steps} steps, solved={solved}, pushes={pushes} -> {path}")
    return path

for name, kw in TESTS.items():
    display(Video(record_test(name, kw), embed=True, width=420))

### Maze diagnostics: from the real start, and with the ball placed 2 and 1 cells before the goal
Shows whether the brain gets lost on the way or only fails the final stop. Runs in a separate
process (the renderer must be set up before MuJoCo is loaded).

In [ ]:
if env_name.startswith("maze"):
    !python scripts/watch_maze.py --run {run} --before 2 1 --best --out {WORK}
    for f in sorted(glob.glob(f"{WORK}/diag_*.mp4")):
        print(f); display(Video(f, embed=True, width=420))

## 6. Export everything (runs, videos, curves) as one zip

In [ ]:
import zipfile
out = f"{WORK}/runs_export.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{WORK}/runs/**/*", recursive=True):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, WORK))
    for f in glob.glob(f"{WORK}/*.mp4") + glob.glob(f"{WORK}/*.png"):
        z.write(f, os.path.join("media", os.path.basename(f)))
with zipfile.ZipFile(out) as z:
    names = z.namelist()
print(f"saved {out} ({os.path.getsize(out) / 1e6:.1f} MB): "
      f"{sum(n.startswith('runs/') for n in names)} run files, "
      f"{sum(n.startswith('media/') for n in names)} videos/images")
print("download runs_export.zip from the Output panel")